# Groq GPT-OSS 20B — 50 invoices with shared VietOCR
Pinned production extraction code, same OCR and scoring as prior benchmarks. Credentials are read from Kaggle Secrets.

In [ ]:

from pathlib import Path
import json, os, sys, time, subprocess, hashlib, unicodedata
from kaggle_secrets import UserSecretsClient
try:
    api_key = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception:
    raise RuntimeError("Enable the GROQ_API_KEY Kaggle Secret for this notebook, then run again.") from None
assert api_key, "The GROQ_API_KEY Kaggle Secret is empty."
os.environ["GROQ_API_KEY"] = api_key
del api_key
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai==2.41.0", "pydantic==2.11.7"],check=True)
from urllib.request import urlopen
commit='0f9c2c19f21c79d45db813d53e8c7fc2bf78cb4f'
repo=Path("/tmp/groq-invoice-source")
package=repo/"document_processing"
package.mkdir(parents=True,exist_ok=True)
(package/"__init__.py").write_text("")
url=f"https://raw.githubusercontent.com/Lenhan231/enterprise-knowledge-ai-agent/{commit}/src/document_processing/invoice_text_processor.py"
with urlopen(url,timeout=30) as response:
    module_source=response.read()
(package/"invoice_text_processor.py").write_bytes(module_source)
sys.path.insert(0,str(repo))
from document_processing.invoice_text_processor import InvoiceTextProcessor, InvoiceFields
ocr_paths=list(Path("/kaggle/input").rglob("ocr_results.json"))
assert len(ocr_paths)==1, "Expected exactly one shared OCR dataset"
raw_ocr=ocr_paths[0].read_bytes()
ocr_data=json.loads(raw_ocr)
samples=ocr_data["samples"]
assert len(samples)==50 and all(row["status"]=="success" for row in samples.values())
print("Shared OCR samples:",len(samples),"| source commit:",commit,flush=True)


In [ ]:
def norm(value, numeric=False):
    if value is None:
        return ""
    text = unicodedata.normalize("NFC", " ".join(str(value).split())).casefold()
    return "".join(char for char in text if char.isdigit()) if numeric else text

def norm_date(value):
    text = norm(value)
    return text.removeprefix("ngày ").removeprefix("ngay ")

def edit_distance(left: str, right: str) -> int:
    previous = list(range(len(right) + 1))
    for i, left_char in enumerate(left, 1):
        current = [i]
        for j, right_char in enumerate(right, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (left_char != right_char),
            ))
        previous = current
    return previous[-1]

def word_edit_distance(left: list[str], right: list[str]) -> int:
    previous = list(range(len(right) + 1))
    for i, left_word in enumerate(left, 1):
        current = [i]
        for j, right_word in enumerate(right, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (left_word != right_word),
            ))
        previous = current
    return previous[-1]

def score(provider_rows: list[dict]) -> tuple[list[dict], dict]:
    headers = ("invoice_no", "date", "seller", "buyer", "subtotal", "tax_amount", "total_amount")
    item_fields = ("description", "quantity", "unit", "unit_price", "amount")
    numeric = {"subtotal", "tax_amount", "total_amount", "quantity", "unit_price", "amount"}
    rows = []
    for result in provider_rows:
        expected = result["expected"]
        predicted = result.get("fields") or {}
        expected_items = expected.get("items") or []
        predicted_items = predicted.get("items") or []
        row = {
            "file": result["file"],
            "quality": result["quality"],
            "status": result["status"],
            "schema_valid": result.get("schema_valid", False),
            "parse_error": result.get("parse_error"),
            "field_conflict_count": len(result.get("field_conflicts") or {}),
            "elapsed_seconds": result.get("elapsed_seconds"),
            "item_count_exact": len(expected_items) == len(predicted_items),
        }
        for field in headers:
            normalize = norm_date if field == "date" else lambda value: norm(value, field in numeric)
            row[field + "_exact"] = normalize(expected.get(field)) == normalize(predicted.get(field))
        row_exact = 0
        for index, expected_item in enumerate(expected_items):
            predicted_item = predicted_items[index] if index < len(predicted_items) else {}
            row_exact += int(all(
                norm(expected_item.get(field), field in numeric) == norm(predicted_item.get(field), field in numeric)
                for field in item_fields
            ))
        row["item_rows_exact_rate"] = row_exact / len(expected_items) if expected_items else float(not predicted_items)
        for field in item_fields:
            row["item_" + field + "_accuracy"] = (
                sum(
                    norm(item.get(field), field in numeric) == norm((predicted_items[i] if i < len(predicted_items) else {}).get(field), field in numeric)
                    for i, item in enumerate(expected_items)
                ) / len(expected_items) if expected_items else 1.0
            )
        row["whole_invoice_exact"] = (
            row["schema_valid"] and not row["parse_error"] and not row["field_conflict_count"]
            and all(row[field + "_exact"] for field in headers)
            and row["item_count_exact"] and row_exact == len(expected_items)
        )
        rows.append(row)

    def mean(values):
        values = list(values)
        return sum(values) / len(values) if values else 0.0

    successes = [row for row in provider_rows if row["status"] == "success"]
    latency = sorted(row["elapsed_seconds"] for row in successes)
    description_char_errors = description_chars = description_word_errors = description_words = 0
    for result in provider_rows:
        predicted = (result.get("fields") or {}).get("items") or []
        for index, expected_item in enumerate(result["expected"].get("items") or []):
            expected_text = norm(expected_item.get("description"))
            predicted_text = norm((predicted[index] if index < len(predicted) else {}).get("description"))
            description_char_errors += edit_distance(expected_text, predicted_text)
            description_chars += len(expected_text)
            expected_tokens, predicted_tokens = expected_text.split(), predicted_text.split()
            description_word_errors += word_edit_distance(expected_tokens, predicted_tokens)
            description_words += len(expected_tokens)
    header_rates = {field: mean(row[field + "_exact"] for row in rows) for field in headers}
    item_rates = {field: mean(row["item_" + field + "_accuracy"] for row in rows) for field in item_fields}
    model_latency = [row["metrics"].get("inference_seconds", 0) for row in successes]
    summary = {
        "sample_count": len(provider_rows),
        "success_count": len(successes),
        "coverage": len(successes) / len(provider_rows) if provider_rows else 0,
        "schema_valid_rate": mean(row["schema_valid"] for row in rows),
        "parse_error_count": sum(bool(row["parse_error"]) for row in rows),
        "field_conflict_count": sum(row["field_conflict_count"] for row in rows),
        "header_exact_match_rates": header_rates,
        "item_count_exact_rate": mean(row["item_count_exact"] for row in rows),
        "item_row_exact_accuracy": mean(row["item_rows_exact_rate"] for row in rows),
        "item_field_accuracy": item_rates,
        "description_cer": description_char_errors / description_chars if description_chars else None,
        "description_wer": description_word_errors / description_words if description_words else None,
        "whole_invoice_exact_rate": mean(row["whole_invoice_exact"] for row in rows),
        "end_to_end_latency_seconds": {
            "mean": mean(row["elapsed_seconds"] for row in successes),
            "median": (latency[len(latency)//2] if len(latency) % 2 else (latency[len(latency)//2-1] + latency[len(latency)//2]) / 2) if latency else None,
            "p95": latency[min(len(latency)-1, int(0.95 * (len(latency)-1)))] if latency else None,
            "model_inference_seconds_mean": mean(model_latency),
        },
        "usage": {
            "prompt_tokens": sum(int(row["metrics"].get("prompt_tokens") or 0) for row in successes),
            "generated_tokens": sum(int(row["metrics"].get("generated_tokens") or 0) for row in successes),
            "requests": sum(int(row["metrics"].get("request_count") or 0) for row in successes),
        },
        "by_quality": {},
    }
    for quality in sorted({row["quality"] for row in rows}):
        group = [row for row in rows if row["quality"] == quality]
        summary["by_quality"][quality] = {
            "sample_count": len(group),
            "coverage": mean(row["status"] == "success" for row in group),
            "whole_invoice_exact_rate": mean(row["whole_invoice_exact"] for row in group),
            "item_field_accuracy": {field: mean(row["item_" + field + "_accuracy"] for row in group) for field in item_fields},
        }
    return rows, summary

In [ ]:

import httpx, openai, re
last_request = [0.0]
token_budget = {"remaining": None, "reset_at": 0.0}
def duration_seconds(value):
    parts=re.findall(r"([\d.]+)(ms|h|m|s)", value or "")
    units={"ms":0.001,"s":1,"m":60,"h":3600}
    return sum(float(n)*units[unit] for n,unit in parts)
def pace_request(request):
    delay=4.0-(time.monotonic()-last_request[0])
    estimated=len(request.content)/2.5 + json.loads(request.content).get("max_completion_tokens",2048)
    if token_budget["remaining"] is not None and estimated>token_budget["remaining"]:
        delay=max(delay,token_budget["reset_at"]-time.monotonic()+1)
    if delay>0:
        time.sleep(delay)
    last_request[0]=time.monotonic()
def record_limits(response):
    remaining=response.headers.get("x-ratelimit-remaining-tokens")
    if remaining is not None:
        token_budget["remaining"]=int(remaining)
        token_budget["reset_at"]=time.monotonic()+duration_seconds(response.headers.get("x-ratelimit-reset-tokens",""))
original_openai=openai.OpenAI
class PacedGroqClient(original_openai):
    def __init__(self, **kwargs):
        kwargs["http_client"]=httpx.Client(event_hooks={"request":[pace_request],"response":[record_limits]})
        super().__init__(**kwargs)
openai.OpenAI=PacedGroqClient


In [ ]:

out_dir=Path("/kaggle/working/groq20b-invoice-50")
out_dir.mkdir(parents=True,exist_ok=True)
output_path=out_dir/"groq20b_results.json"
data={
    "metadata":{
        "sample_count":50,"model_id":"openai/gpt-oss-20b","provider":"groq",
        "source_commit":commit,"prompt_source_sha256":hashlib.sha256(module_source).hexdigest(),
        "ocr_sha256":hashlib.sha256(raw_ocr).hexdigest(),
        "ocr_repo":ocr_data["metadata"].get("ocr_repo"),"ocr_revision":ocr_data["metadata"].get("ocr_revision"),
        "shared_ocr_inputs":True,"reasoning_effort":"low","response_format":"json_object",
        "pacing":"4s minimum between requests, wait for Groq token-reset headers when remaining budget is low",
        "latency_note":"elapsed_seconds excludes cached OCR and includes API quota pacing/retries",
    },
    "provider":"groq20b","results":{},
}
processor=InvoiceTextProcessor({"invoice_provider":"groq","invoice_timeout_seconds":60})
for index,(filename,sample) in enumerate(samples.items(),1):
    row={"file":filename,"invoice_no":sample["invoice_no"],"quality":sample["quality"],
         "expected":sample["expected"],"status":"failed"}
    started=time.perf_counter()
    try:
        result=processor.process(sample.get("elements") or [],sample.get("tables") or [])
        fields=result["fields"]
        InvoiceFields.model_validate(fields)
        row.update(status="success",schema_valid=True,fields=fields,model=result["model"],
                   revision=result["revision"],metrics=result["metrics"],parse_error=result["parse_error"],
                   field_conflicts=result["field_conflicts"],raw_response=result["raw_response"])
    except Exception as error:
        row.update(error=type(error).__name__+": "+str(error)[:250],schema_valid=False)
    row["elapsed_seconds"]=round(time.perf_counter()-started,2)
    data["results"][filename]=row
    scored,metrics=score(list(data["results"].values()))
    data.update(scores=scored,metrics=metrics)
    tmp=output_path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding="utf-8")
    tmp.replace(output_path)
    print(f'GROQ20B {index}/50 {filename} status={row["status"]} parse_error={row.get("parse_error")} elapsed={row["elapsed_seconds"]}s whole_exact={metrics["whole_invoice_exact_rate"]:.1%}',flush=True)
print("FINAL_METRICS",json.dumps(data["metrics"]),flush=True)
